# ⚡ ComfyUI Quick Launch Colab

### 🔴 Author :  **Dhanush Kannan**
#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **VFX Artist · AI Enthusiast**
---

### ✨ Features
- 🔗 **ngrok Tunnel** — Stable & reliable
- 💾 **Drive Symlinks** — Models & nodes saved permanently. Zero re-downloading
- 📥 **Smart Downloader** — Checkpoints, Flux GGUF, CLIP, VAE, LoRA, ControlNet & more
- 🧩 **ComfyUI Manager** — Install & manage custom nodes easily
- 🔊 **Anti-Disconnect** — Silent audio keeps your session alive
- 🎛️ **VRAM Profiles** — Auto · Low VRAM (T4) · High VRAM (A100)
- ☁️ **Cloudflare Fallback** — Auto-switches if ngrok ever fails
- ⚡ **Parallel Node Restore** — Custom node packages reinstalled faster every session

---

### 🚀 Quick Start
1. Runtime → Change runtime type → **T4 GPU**
2. Get your **free ngrok token** → [ngrok.com](https://ngrok.com)
3. Paste token in **Cell 1** → Run cells top to bottom
4. Click the **ngrok link** in Cell 4 → ComfyUI opens! 🎉

> ⚠️ *If link shows 403 — click the address bar and press Enter once. Opens instantly!*

> 💡 *Models stay in Drive forever. Next session — just run Cell 1 and Cell 4 only!*

In [ ]:
#@title ⚙️ 1. System Initialization
#@markdown Configure your settings below, then run this cell.
#@markdown ---
#@markdown > ⚠️ **IMPORTANT:** After your first run — **uncheck** `UPDATE_COMFY_UI` and `INSTALL_COMFYUI_MANAGER` to save time on future sessions!

import os
import shutil
import subprocess
import glob
import concurrent.futures
from google.colab import drive

# --- YOUR NGROK TOKEN ---
NGROK_TOKEN = "3GqOfniJRIBlJUQhdjgDpc2svaH_7QPPnHKRU13Ly79cogBM9"  #@param {type:"string"}

# --- OPTIONS ---
UPDATE_COMFY_UI         = True  #@param {type:"boolean"}
INSTALL_COMFYUI_MANAGER = True   #@param {type:"boolean"}

# --- PATHS ---
LOCAL_WORKSPACE = "/content/ComfyUI"
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI"

# 1. Mount Drive
print("💾 Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Clone or Update ComfyUI locally
if not os.path.exists(LOCAL_WORKSPACE):
    print("📦 Cloning ComfyUI...")
    !git clone -q https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE}
else:
    if UPDATE_COMFY_UI:
        print("🔄 Updating ComfyUI...")
        lock = os.path.join(LOCAL_WORKSPACE, ".git", "index.lock")
        if os.path.exists(lock): os.remove(lock)
        !cd {LOCAL_WORKSPACE} && git pull -q
    else:
        print("✅ ComfyUI already installed.")

# 3. Symlink heavy folders to Drive
print("\n🔗 Linking Drive storage...")
FOLDERS = ["models", "custom_nodes", "output", "input", "user"]
for folder in FOLDERS:
    local  = os.path.join(LOCAL_WORKSPACE, folder)
    drive_ = os.path.join(DRIVE_WORKSPACE, folder)
    os.makedirs(drive_, exist_ok=True)
    if os.path.exists(local) and not os.path.islink(local):
        shutil.rmtree(local)
    if not os.path.exists(local):
        os.symlink(drive_, local)
    print(f"   🔗 {folder} → Drive")

# 4. Install ComfyUI Manager
if INSTALL_COMFYUI_MANAGER:
    manager_path = os.path.join(LOCAL_WORKSPACE, "custom_nodes", "ComfyUI-Manager")
    if not os.path.exists(manager_path):
        print("\n📦 Installing ComfyUI Manager...")
        !git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git {manager_path}
    else:
        print("\n✅ ComfyUI Manager found. Updating...")
        !cd {manager_path} && git pull -q

# 5. Install Dependencies
print("\n🛠️ Installing dependencies...")
!cd {LOCAL_WORKSPACE} && pip install -q xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q "sqlalchemy>=2.0" aiosqlite --upgrade

# 6. Restore Custom Node Packages (Parallel - Faster)
print("\n📦 Restoring custom node packages...")

req_files = glob.glob(f"{LOCAL_WORKSPACE}/custom_nodes/*/requirements.txt")

def install_req(req):
    node_name = req.split("/")[-2]
    subprocess.run(["pip", "install", "-q", "-r", req],
                   capture_output=True)
    return node_name

if req_files:
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(install_req, req): req for req in req_files}
        for future in concurrent.futures.as_completed(futures):
            node_name = future.result()
            print(f"   ✅ Restored: {node_name}")
    print("✅ All custom node packages restored!")
else:
    print("   No custom nodes found yet.")

# 7. Save ngrok token for Cell 4
os.environ['NGROK_TOKEN'] = NGROK_TOKEN

print("\n✅ System ready! Run the next cell.")

In [ ]:
#@title 📥 2. Smart Model Downloader
#@markdown Paste direct download links below. Separate multiple URLs by comma or new line.
#@markdown Already downloaded files are skipped automatically.

import os, re, requests
from urllib.parse import urlparse, unquote
from tqdm.auto import tqdm
import subprocess

WORKSPACE = "/content/ComfyUI"

# --- Paste Your URLs Here ---
CHECKPOINT_URLS      = ""  #@param {type:"string"}
UNET_DIFFUSION_URLS  = "https://huggingface.co/Comfy-Org/Krea-2/blob/main/diffusion_models/krea2_turbo_fp8_scaled.safetensors"  #@param {type:"string"}
TEXT_ENCODER_URLS    = ""  #@param {type:"string"}
CLIP_VISION_URLS     = ""  #@param {type:"string"}
VAE_URLS             = ""  #@param {type:"string"}
LORA_URLS            = ""  #@param {type:"string"}
CONTROLNET_URLS      = ""  #@param {type:"string"}
UPSCALE_MODELS_URLS  = ""  #@param {type:"string"}
EMBEDDING_URLS       = ""  #@param {type:"string"}
CUSTOM_NODE_URLS     = ""  #@param {type:"string"}

# --- Directory Map ---
DIRS = {
    "checkpoints"   : f"{WORKSPACE}/models/checkpoints",
    "unet"          : f"{WORKSPACE}/models/diffusion_models",
    "clip"          : f"{WORKSPACE}/models/clip",
    "clip_vision"   : f"{WORKSPACE}/models/clip_vision",
    "vae"           : f"{WORKSPACE}/models/vae",
    "loras"         : f"{WORKSPACE}/models/loras",
    "controlnet"    : f"{WORKSPACE}/models/controlnet",
    "upscale_models": f"{WORKSPACE}/models/upscale_models",
    "embeddings"    : f"{WORKSPACE}/models/embeddings",
    "custom_nodes"  : f"{WORKSPACE}/custom_nodes",
}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

def get_filename(url, response):
    if "Content-Disposition" in response.headers:
        m = re.findall('filename="?([^"]+)"?', response.headers["Content-Disposition"])
        if m: return m[0]
    name = unquote(os.path.basename(urlparse(url).path))
    return name or "model.safetensors"

def download_file(url, target_dir):
    try:
        r = requests.get(url, stream=True, allow_redirects=True)
        r.raise_for_status()
        fname = get_filename(url, r)
        fpath = os.path.join(target_dir, fname)
        if os.path.exists(fpath):
            print(f"   ⏩ Skipping (exists): {fname}")
            return
        total = int(r.headers.get('content-length', 0))
        print(f"   📥 Downloading: {fname}")
        with tqdm(total=total, unit='B', unit_scale=True, unit_divisor=1024, dynamic_ncols=True) as bar:
            with open(fpath, 'wb') as f:
                for chunk in r.iter_content(1024*1024):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))
        print(f"   ✅ Done: {fname}")
    except Exception as e:
        print(f"   ❌ Failed: {url} — {e}")

def process(urls_str, target_dir, is_node=False):
    if not urls_str.strip(): return
    urls = [u.strip() for u in urls_str.replace(',', '\n').split('\n') if u.strip()]
    print(f"\n📂 {os.path.basename(target_dir)} — {len(urls)} item(s)")
    for url in urls:
        if is_node:
            name = url.split('/')[-1].replace('.git', '')
            path = os.path.join(target_dir, name)
            if not os.path.exists(path):
                print(f"   ⬇️ Cloning: {name}")
                subprocess.run(["git", "clone", "-q", url, path])
                req = os.path.join(path, "requirements.txt")
                if os.path.exists(req):
                    subprocess.run(["pip", "install", "-q", "-r", req])
                print(f"   ✅ Installed: {name}")
            else:
                print(f"   ⏩ Already exists: {name}")
        else:
            download_file(url, target_dir)

process(CHECKPOINT_URLS,     DIRS["checkpoints"])
process(UNET_DIFFUSION_URLS, DIRS["unet"])
process(TEXT_ENCODER_URLS,   DIRS["clip"])
process(CLIP_VISION_URLS,    DIRS["clip_vision"])
process(VAE_URLS,            DIRS["vae"])
process(LORA_URLS,           DIRS["loras"])
process(CONTROLNET_URLS,     DIRS["controlnet"])
process(UPSCALE_MODELS_URLS, DIRS["upscale_models"])
process(EMBEDDING_URLS,      DIRS["embeddings"])
process(CUSTOM_NODE_URLS,    DIRS["custom_nodes"], is_node=True)

print("\n🎉 All downloads complete!")

In [ ]:
#@title 🔊 3. Anti-Disconnect
from IPython.display import display, HTML
display(HTML('''
<i>Silent audio loop — keeps your tab active while ComfyUI runs.</i><br>
<audio src="https://raw.githubusercontent.com/anars/blank-audio/master/10-minutes-of-silence.mp3"
       autoplay loop controls style="width:300px;margin-top:8px" />
'''))

In [ ]:
#@title 🚀 4. Launch ComfyUI
#@markdown Select your GPU memory profile below.

import os, subprocess, threading, time, socket

# Force SQLAlchemy upgrade before ComfyUI starts
subprocess.run(["pip", "install", "-q", "sqlalchemy>=2.0", "aiosqlite", "--upgrade"],
               capture_output=True)

# Reload sqlalchemy in current session
import importlib, sys
if 'sqlalchemy' in sys.modules:
    del sys.modules['sqlalchemy']

# --- Settings ---
MEMORY_PROFILE          = "Standard (Auto-Detect)"  #@param ["Standard (Auto-Detect)", "Low VRAM (T4 — Heavy Models)", "High VRAM (A100 Only)"]
LIVE_PREVIEWS           = True                       #@param {type:"boolean"}
USE_CLOUDFLARE_FALLBACK = True                       #@param {type:"boolean"}

WORKSPACE = "/content/ComfyUI"
PORT      = 8188

# --- Build launch args ---
ARGS = "--dont-print-server --listen 127.0.0.1 --port 8188"
if "Low VRAM"   in MEMORY_PROFILE: ARGS += " --lowvram"
elif "High VRAM" in MEMORY_PROFILE: ARGS += " --highvram"
if LIVE_PREVIEWS: ARGS += " --preview-method auto"

# --- Get ngrok token ---
NGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')
if not NGROK_TOKEN or NGROK_TOKEN == 'paste_your_ngrok_token_here':
    raise ValueError("❌ Ngrok token missing! Paste your token in Cell 1 and re-run from the top.")

# Install ngrok
print("📦 Setting up ngrok...")
!pip install -q pyngrok
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN

# Install Cloudflare fallback
if USE_CLOUDFLARE_FALLBACK:
    print("📦 Setting up Cloudflare (fallback)...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb -q

def wait_for_comfy(port):
    while True:
        time.sleep(1)
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if s.connect_ex(('127.0.0.1', port)) == 0:
            s.close()
            break
        s.close()

    print("\n🟢 ComfyUI is running. Launching tunnel...\n")

    # Try ngrok first
    try:
      from pyngrok import ngrok, conf
      conf.get_default().auth_token = NGROK_TOKEN
      ngrok.set_auth_token(NGROK_TOKEN)

      tunnel = ngrok.connect(port, "http")
      base_url = tunnel.public_url.replace("http://", "https://")
      full_url = f"{base_url}?ngrok-skip-browser-warning=true"

      print("\n" + "="*60)
      print(f"🔗 COMFYUI LINK (ngrok): {full_url}")
      print("="*60)
      print("👆 Click the link above to open ComfyUI")
      print("✅ ngrok tunnel active — stable & reliable\n")
    except Exception as e:
      print(f"⚠️ ngrok failed: {e}")

      if USE_CLOUDFLARE_FALLBACK:
        print("🔄 Trying Cloudflare as fallback...")
        p = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
        for line in p.stderr:
            l = line.decode()
            if "trycloudflare.com" in l:
                url = l[l.find("http"):].strip()
                print("\n" + "="*60)
                print(f"🔗 COMFYUI LINK (Cloudflare fallback): {url}")
                print("="*60 + "\n")
                break
        else:
          print("❌ No tunnel available. Check your ngrok token in Cell 1.")

threading.Thread(target=wait_for_comfy, daemon=True, args=(PORT,)).start()

# Launch ComfyUI
if os.path.exists(os.path.join(WORKSPACE, "main.py")):
    %cd {WORKSPACE}
    print(f"🚀 Launching ComfyUI [{MEMORY_PROFILE}]...")
    print("⏳ Wait for the link to appear below...\n")
    !python main.py {ARGS}
else:
    print("❌ main.py not found. Did Cell 1 complete successfully?")